# TEST 2 avec word to vec + tfidf

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

print('Numpy:', np.__version__)
print('Pandas:', pd.__version__)

Numpy: 2.4.2
Pandas: 3.0.1


In [4]:
csv_path = "../Student_Dataset.csv"
dataframe = pd.read_csv(csv_path, encoding='utf-8-sig')
print(dataframe.head())
print(dataframe.shape)
print(dataframe.info())

                                                   1  \
2                                           0x000000   
3                                           0x000000   
4  A short sprint brought on a twinge in the midd...   
5                                           0x000000   
6  A sudden twist while reaching for a bag made o...   

  On uneven paths the heel feels wobbly side‑to‑side. Slanted pavements bring a quick aching shift under the ankle when I step.  
2  The feverish feeling eased, but I’m now breath...                                                                             
3  While working in the garden, I touched a metal...                                                                             
4                                                NaN                                                                             
5  I accidentally left my hand on a hot mug for a...                                                                             
6            

In [6]:
csv_path = "../Student_Dataset.csv"

tempdf = pd.read_csv(csv_path, encoding='utf-8-sig', header=None,
    names=['id', 'col2', 'col3'])

tempdf['text'] = np.where(tempdf['col2'] == '0x000000', tempdf['col3'], tempdf['col2'])
dataframe = tempdf[['id', 'text']]



print(dataframe.head())
print(dataframe.shape)
print(dataframe.info())
print('nombres elements louche dans le df: ', dataframe['text'].str.contains('0x000000').sum())


   id                                               text
0   1  On uneven paths the heel feels wobbly side‑to‑...
1   2  The feverish feeling eased, but I’m now breath...
2   3  While working in the garden, I touched a metal...
3   4  A short sprint brought on a twinge in the midd...
4   5  I accidentally left my hand on a hot mug for a...
(1011, 2)
<class 'pandas.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      1011 non-null   int64
 1   text    1011 non-null   str  
dtypes: int64(1), str(1)
memory usage: 15.9 KB
None
nombres elements louche dans le df:  0


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

vectorizer = TfidfVectorizer(stop_words='english')
stop_words = vectorizer.get_stop_words()

def remove_stop_words(text):
    tokens = text.lower().split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

import re

def remove_numbers(text):
    return re.sub(r'\d+\.?\d*\S*', '', text)



dataframe['clean_text'] = dataframe['text'].apply(remove_stop_words)
dataframe['clean_text'] = dataframe['clean_text'].apply(remove_numbers)

print(dataframe[['text', 'clean_text']].head(3))


                                                text  \
0  On uneven paths the heel feels wobbly side‑to‑...   
1  The feverish feeling eased, but I’m now breath...   
2  While working in the garden, I touched a metal...   

                                          clean_text  
0  uneven paths heel feels wobbly side‑to‑side. s...  
1  feverish feeling eased, i’m breathing faster d...  
2  working garden, touched metal grill stove. lef...  


ICI on inclus le fait de plustipliquer chaque embedding de mot par TFIDF afin d'avoir des embeddings pondéré  et pouvoir ensnuite possiblement faire un PCA significative .

refelctions : La pCA ne se fait elle pas indirectement lorsque la ponderation exerce une moyen passive ou les éléments non discriminants sont deja réduits au silence sans pour autnant etre supprimé ? 

In [8]:
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. TF-IDF
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(dataframe['clean_text'])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

# 2. Word2Vec
tokenized = dataframe['clean_text'].apply(str.split).tolist()
w2v_model = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, workers=4, epochs=50)

# 3. Vecteurs documents = moyenne pondérée W2V par TF-IDF
def get_pond_vector(doc_idx):
    row = tfidf_matrix[doc_idx]
    weighted_vec = np.zeros(w2v_model.vector_size)
    total_weight = 0
    for col_idx in row.nonzero()[1]:
        word = tfidf_feature_names[col_idx]
        tfidf_weight = row[0, col_idx]
        if word in w2v_model.wv:
            weighted_vec += tfidf_weight * w2v_model.wv[word]
            total_weight += tfidf_weight
    if total_weight > 0:
        weighted_vec /= total_weight
    return weighted_vec

tfidf_w2v_matrix = np.array([get_pond_vector(i) for i in range(tfidf_matrix.shape[0])])

print("TF-IDF shape:", tfidf_matrix.shape)
print("TF-IDF x W2V shape:", tfidf_w2v_matrix.shape)


TF-IDF shape: (1011, 2130)
TF-IDF x W2V shape: (1011, 100)


on constate la réduction de dimension dont je parle mais c'tes différents car la reduction de dimenssion ce fait en gardant la rela tions de smots et ne regardant pas la variance explqiué ...;